In [0]:
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, round, avg, sum, count, when, lit,
    date_format, month, year, current_timestamp
)

#read from silver
timesheets_df = spark.table("silver.timesheets")
employees_df  = spark.table("silver.employees")
projects_df   = spark.table("silver.projects")

print(f"Silver timesheets: {timesheets_df.count()}")
print(f"Silver employees:  {employees_df.count()}")
print(f"Silver projects:   {projects_df.count()}")

In [0]:
# utilisation summary 
from pyspark.sql.functions import trunc

#join timesheets to employees and projects
joined_df = (
    timesheets_df
    .join(employees_df.select("employee_id", "name", "seniority", "practice_area"), 
          on="employee_id", how="left")
    .join(projects_df.select("project_id", "project_type", "client"),
          on="project_id", how="left")
)

#monthly utilisation per employee
monthly_util_df = (
    joined_df
    .withColumn("month", date_format(col("week_start"), "yyyy-MM"))
    .groupBy("employee_id", "name", "seniority", "practice_area", "month")
    .agg(
        round(sum("hours_logged_clean"), 2).alias("total_hours"),
        round(avg("utilisation_pct"), 2).alias("avg_utilisation_pct"),
        sum(when(col("billable") == True, col("hours_logged_clean"))
            .otherwise(lit(0))).alias("billable_hours"),
        sum(when(col("billable") == False, col("hours_logged_clean"))
            .otherwise(lit(0))).alias("non_billable_hours"),
        count("timesheet_id").alias("timesheet_entries")
    )
    #flag over and under utilisation
    .withColumn("utilisation_flag", when(
        col("avg_utilisation_pct") >= 100, lit("Over utilised"))
        .when(col("avg_utilisation_pct") >= 70, lit("On target"))
        .when(col("avg_utilisation_pct") >= 40, lit("Under utilised"))
        .otherwise(lit("Critically under utilised")))
    .withColumn("billable_pct", round(
        (col("billable_hours") / col("total_hours")) * 100, 2))
    .withColumn("gold_loaded_at", current_timestamp())
)

print(f"Monthly utilisation rows: {monthly_util_df.count()}")
monthly_util_df.show(5)

In [0]:
#  Practice area summary 
practice_summary_df = (
    monthly_util_df
    .groupBy("practice_area", "month")
    .agg(
        round(avg("avg_utilisation_pct"), 2).alias("avg_utilisation_pct"),
        round(sum("total_hours"), 2).alias("total_hours"),
        round(sum("billable_hours"), 2).alias("total_billable_hours"),
        round(sum("non_billable_hours"), 2).alias("total_non_billable_hours"),
        count("employee_id").alias("headcount"),
        count(when(col("utilisation_flag") == "Over utilised", True)).alias("over_utilised_count"),
        count(when(col("utilisation_flag") == "On target", True)).alias("on_target_count"),
        count(when(col("utilisation_flag") == "Under utilised", True)).alias("under_utilised_count"),
        count(when(col("utilisation_flag") == "Critically under utilised", True)).alias("critical_count")
    )
    .withColumn("gold_loaded_at", current_timestamp())
)

print(f"Practice area summary rows: {practice_summary_df.count()}")
practice_summary_df.show(5)

# ── Write to Gold ─────────────────────────────────────────────────────────────
monthly_util_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.utilisation_monthly")
practice_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.utilisation_by_practice")

print(f"\nWritten to gold.utilisation_monthly:     {spark.table('gold.utilisation_monthly').count()} rows")
print(f"Written to gold.utilisation_by_practice: {spark.table('gold.utilisation_by_practice').count()} rows")